### Initialize variables and imports

In [4]:
import autostore

from automol import Geometry
from autostore import (
    Calculation,
    CalculationRow,
    CalculationGeometryLink,
    Database,
    EnergyRow,
    GeometryRow,
    Role,
    utils,
)
from qccompute import compute
from qcdata import CalcType

db = Database("example.db")

geo = Geometry(symbols=["O", "H", "H"], coordinates=[[0, 0, 0], [1, 0, 0], [0, 1, 0]])  # ty:ignore[invalid-argument-type]
calc = Calculation(program="crest", method="gfnff")

### Resolve existing calculations & geometries

In [ ]:
matched = False
# --- Resolve geometry ---
inp_geo_rows = db.find_or_add(
    row=GeometryRow.from_geometry(geo), eager_load=False,
)
inp_geo_row = autostore.utils.verify_single_iteration(inp_geo_rows)

# --- Resolve calculation ---
inp_calc_row = CalculationRow.from_calculation(calc)
inp_calc_row.calc_type = CalcType.energy

existing_calc_rows = db.find(
    row=inp_calc_row, eager_load=True, exclude_id=True,
)

# Find matching energy rows
for calc_row in existing_calc_rows:
    if not calc_row.energies:
        continue
    for link in calc_row.geometry_links:
        if link.geometry_id == inp_geo_row.id and link.role == Role.input:
            for ene_row in calc_row.energies:
                if ene_row.value:
                    matched = True
                    print(f"{ene_row = }")

ene_row = EnergyRow(calculation_id=1, geometry_id=1, id=1, value=-0.320207546368996)


### Compute

In [6]:
if not matched:
    prog = inp_calc_row.super_program or inp_calc_row.program
    prog_inp = inp_calc_row.program_input(input_geo=inp_geo_row)
    prog_out = compute(prog, prog_inp)

    # Instantiate output CalcRow (with Provenance) from prog_out
    out_calc_row = CalculationRow.from_program_output(prog_out=prog_out)
    db.add(row=out_calc_row)  # Adds and updates with row ID

    calc_geo_link = CalculationGeometryLink(
        geometry_id=inp_geo_row.id, calculation_id=out_calc_row.id, role=Role.input
    )
    db.add(row=calc_geo_link)

    ene_row = EnergyRow(
        calculation_id=out_calc_row.id,
        geometry_id=inp_geo_row.id,
        value=prog_out.data.energy,
    )
    db.add(row=ene_row)